# StayNest - Session 7 Assignment (Delta Lake & Lakehouse)
Work through the 8 tasks in order. Read the Assignment Questions PDF for the full
detail and acceptance criteria. Fill each `# TODO` cell, run it, and keep the output
visible. Runs on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `bookings_updates.csv` to a Volume, set `BASE`,
`CATALOG`, `SCHEMA`, and run this cell. Expect 12000 / 200 / 200.

In [0]:
BASE    = "/Volumes/workspace/default/staynest"
CATALOG = "workspace"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

bookings: 12000, hotels: 200, updates: 200


## Task 1 - Read the plan and force a broadcast join
Join bookings to hotels and call `.explain()` to see the plan. Then force a
broadcast join with `broadcast(hotels_df)` and `.explain()` again. In a comment,
say which join each plan used and why broadcast avoids a shuffle.
(Tip: hotels also has a `city` column, so `hotels_df.drop("city")` before joining.)

In [0]:
# TODO
from pyspark.sql.functions import col

bookings_hotels_df = bookings_df.alias("b").join(hotels_df.alias("h") , col("b.hotel_id") == col("h.hotel_id")
                                                 ,"inner").drop(col("h.city"),col("h.hotel_id"))

#bookings_hotels_df.display()

In [0]:
bookings_hotels_df.explain(True)

== Parsed Logical Plan ==
Project [booking_id#13419, customer_id#13420, hotel_id#13421, booking_date#13422, city#13423, nights#13424, amount#13425, status#13426, hotel_name#13451, category#13453, star_rating#13454]
+- Join Inner, (hotel_id#13421 = hotel_id#13450)
   :- SubqueryAlias b
   :  +- Relation [booking_id#13419,customer_id#13420,hotel_id#13421,booking_date#13422,city#13423,nights#13424,amount#13425,status#13426] csv
   +- SubqueryAlias h
      +- Relation [hotel_id#13450,hotel_name#13451,city#13452,category#13453,star_rating#13454] csv

== Analyzed Logical Plan ==
booking_id: int, customer_id: int, hotel_id: int, booking_date: date, city: string, nights: int, amount: double, status: string, hotel_name: string, category: string, star_rating: double
Project [booking_id#13419, customer_id#13420, hotel_id#13421, booking_date#13422, city#13423, nights#13424, amount#13425, status#13426, hotel_name#13451, category#13453, star_rating#13454]
+- Join Inner, (hotel_id#13421 = hotel_id#13

In [0]:
from pyspark.sql.functions import broadcast

broadcast_bookings_hotels_df = bookings_df.alias("b").join(broadcast(hotels_df.alias("h")) , col("b.hotel_id") == col("h.hotel_id"),"inner").drop(col("h.city"),col("h.hotel_id"))

broadcast_bookings_hotels_df.explain(True)

== Parsed Logical Plan ==
Project [booking_id#13419, customer_id#13420, hotel_id#13421, booking_date#13422, city#13423, nights#13424, amount#13425, status#13426, hotel_name#13451, category#13453, star_rating#13454]
+- Join Inner, (hotel_id#13421 = hotel_id#13450)
   :- SubqueryAlias b
   :  +- Relation [booking_id#13419,customer_id#13420,hotel_id#13421,booking_date#13422,city#13423,nights#13424,amount#13425,status#13426] csv
   +- ResolvedHint (strategy=broadcast)
      +- SubqueryAlias h
         +- Relation [hotel_id#13450,hotel_name#13451,city#13452,category#13453,star_rating#13454] csv

== Analyzed Logical Plan ==
booking_id: int, customer_id: int, hotel_id: int, booking_date: date, city: string, nights: int, amount: double, status: string, hotel_name: string, category: string, star_rating: double
Project [booking_id#13419, customer_id#13420, hotel_id#13421, booking_date#13422, city#13423, nights#13424, amount#13425, status#13426, hotel_name#13451, category#13453, star_rating#13454

Before : in this case spark automatically chose broadcast join instead of sort-merge join. <br>
After : added force broadcast join.<br>
With broadcast(hotels_df), using BroadcastHashJoin,  small hotel table are sent to all executors. Since each executor has<br>
 the full small table, no shuffle is needed thus making the join faster.

## Task 2 - Create a Delta table, then read its history
Write `bookings_df` as a managed Delta table with `saveAsTable`. Then create some
history: run an `UPDATE` (set pending to completed) and a `DELETE` (remove
cancelled). Show `DESCRIBE HISTORY` and point out the versioned commits.

In [0]:
# TODO

bookings_df.write.mode("overwrite").format("delta").saveAsTable("workspace.default.bookings")

spark.table("workspace.default.bookings").show()


+----------+-----------+--------+------------+---------+------+--------+---------+
|booking_id|customer_id|hotel_id|booking_date|     city|nights|  amount|   status|
+----------+-----------+--------+------------+---------+------+--------+---------+
|   9000000|     701600|    3095|  2025-11-27|   Jaipur|     4| 6087.65|completed|
|   9000001|     700065|    3057|  2025-11-06|    Delhi|     1| 8211.19|cancelled|
|   9000002|     701392|    3187|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|
|   9000003|     700867|    3112|  2025-03-22|Bengaluru|     5| 7880.62|completed|
|   9000004|     701521|    3043|  2025-04-19|   Mumbai|     5|21021.51|  pending|
|   9000005|     701998|    3018|  2025-10-14|   Mumbai|     2|18124.49|cancelled|
|   9000006|     701336|    3012|  2025-11-25|    Delhi|     7|70999.15|completed|
|   9000007|     700868|    3127|  2025-04-10|   Mumbai|     2|15693.64|completed|
|   9000008|     700687|    3045|  2025-01-15|   Mumbai|     1| 1492.62|completed|
|   

In [0]:
spark.sql(f""" update  workspace.default.bookings 
          set status='completed'
          where status ='pending' """)

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql(f""" delete from workspace.default.bookings 
          where status ='cancelled' """)

DataFrame[num_affected_rows: bigint]

In [0]:
history = spark.sql(f""" describe history workspace.default.bookings """)

display(history)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-17T13:11:31.000Z,72477596197237,i_majumdar@outlook.com,DELETE,"Map(predicate -> [""(status#14352 = cancelled)""])",null,List(3067771948461152),2186cada-a232-4604-94fb-f6b978b302bc,0717-123401-9cj1cu22-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1398, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 994, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 403)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-17T13:11:28.000Z,72477596197237,i_majumdar@outlook.com,UPDATE,"Map(predicate -> [""(status#13929 = pending)""])",null,List(3067771948461152),d97376e0-1309-4ccc-9dd7-ded700d63576,0717-123401-9cj1cu22-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2520, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1067, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 1449)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-17T13:11:23.000Z,72477596197237,i_majumdar@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3067771948461152),e34434d5-6839-4918-916a-7678f517b561,0717-123401-9cj1cu22-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 12000, numOutputBytes -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


## Task 3 - Time travel and RESTORE
Read the table as it was at **version 0** (before your UPDATE and DELETE) and show
its count. Then `RESTORE` the table to version 0 and confirm the count is back.
Show that RESTORE appears as a new commit in the history.

In [0]:
# TODO

version0 =(spark.read.format("delta")
            .option("versionAsOf",0)
            .table("workspace.default.bookings")

)
current =  spark.table("workspace.default.bookings").count()

print(f"Count at version0 : {version0.count()} ")
print(f"Count at current version : {current}")

Count at version0 : 12000 
Count at current version : 10563


In [0]:
spark.sql(f"""
RESTORE table workspace.default.bookings
TO VERSION AS OF 0
"""
)
after_restore =  spark.table("workspace.default.bookings").count()
print(f"Count at after restore  : {after_restore}")

Count at after restore  : 12000


In [0]:
history = spark.sql(f""" describe history workspace.default.bookings """)

display(history)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-07-17T13:11:42.000Z,72477596197237,i_majumdar@outlook.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(3067771948461152),5b457e3c-b770-4ba0-8d5d-4e640bef238a,0717-123401-9cj1cu22-v2n,3,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 101467, numRemovedFiles -> 1, restoredFilesSize -> 113163, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-07-17T13:11:33.000Z,72477596197237,i_majumdar@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3067771948461152),2186cada-a232-4604-94fb-f6b978b302bc,0717-123401-9cj1cu22-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 125764, p25FileSize -> 101467, numDeletionVectorsRemoved -> 1, minFileSize -> 101467, numAddedFiles -> 1, maxFileSize -> 101467, p75FileSize -> 101467, p50FileSize -> 101467, numAddedBytes -> 101467)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-17T13:11:31.000Z,72477596197237,i_majumdar@outlook.com,DELETE,"Map(predicate -> [""(status#14352 = cancelled)""])",null,List(3067771948461152),2186cada-a232-4604-94fb-f6b978b302bc,0717-123401-9cj1cu22-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1398, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 994, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 403)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-17T13:11:28.000Z,72477596197237,i_majumdar@outlook.com,UPDATE,"Map(predicate -> [""(status#13929 = pending)""])",null,List(3067771948461152),d97376e0-1309-4ccc-9dd7-ded700d63576,0717-123401-9cj1cu22-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2520, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1067, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 1449)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-17T13:11:23.000Z,72477596197237,i_majumdar@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3067771948461152),e34434d5-6839-4918-916a-7678f517b561,0717-123401-9cj1cu22-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 12000, numOutputBytes -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


## Task 4 - OPTIMIZE and ZORDER
Run `OPTIMIZE` on your Delta table to compact files. Then run
`OPTIMIZE ... ZORDER BY (city)`. In a comment, say what OPTIMIZE does and why
`city` is a good ZORDER column but `status` would not be.

In [0]:
# TODO

optimize_result = spark.sql(f"OPTIMIZE workspace.default.bookings")


In [0]:
spark.sql(f"""
      OPTIMIZE workspace.default.bookings
      ZORDER BY (city)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

 OPTIMIZE compacts many small files into fewer larger files which improves read performance and file management issue.<br>
 city is a good ZORDER column because queries often filter on specific cities where column like status usually <br>
 have low cardinality , so using ZORDER by it will be less beneficial .


In [0]:
history = spark.sql(f""" describe history workspace.default.bookings """)

display(history)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-07-17T13:11:42.000Z,72477596197237,i_majumdar@outlook.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(3067771948461152),5b457e3c-b770-4ba0-8d5d-4e640bef238a,0717-123401-9cj1cu22-v2n,3,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 101467, numRemovedFiles -> 1, restoredFilesSize -> 113163, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-07-17T13:11:33.000Z,72477596197237,i_majumdar@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3067771948461152),2186cada-a232-4604-94fb-f6b978b302bc,0717-123401-9cj1cu22-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 125764, p25FileSize -> 101467, numDeletionVectorsRemoved -> 1, minFileSize -> 101467, numAddedFiles -> 1, maxFileSize -> 101467, p75FileSize -> 101467, p50FileSize -> 101467, numAddedBytes -> 101467)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-17T13:11:31.000Z,72477596197237,i_majumdar@outlook.com,DELETE,"Map(predicate -> [""(status#14352 = cancelled)""])",null,List(3067771948461152),2186cada-a232-4604-94fb-f6b978b302bc,0717-123401-9cj1cu22-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1398, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 994, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 403)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-17T13:11:28.000Z,72477596197237,i_majumdar@outlook.com,UPDATE,"Map(predicate -> [""(status#13929 = pending)""])",null,List(3067771948461152),d97376e0-1309-4ccc-9dd7-ded700d63576,0717-123401-9cj1cu22-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2520, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1067, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 1449)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-17T13:11:23.000Z,72477596197237,i_majumdar@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3067771948461152),e34434d5-6839-4918-916a-7678f517b561,0717-123401-9cj1cu22-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 12000, numOutputBytes -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


## Task 5 - Bronze: land the raw data
Write the raw bookings to a `bronze_bookings` Delta table, keeping every row and
adding an `ingested_at` timestamp column.

In [0]:
# TODO
from pyspark.sql.functions import current_timestamp

Base = "/Volumes/workspace/default/staynest"

bronze_ingestion = (spark.read.csv(f"{Base}/bookings.csv", header =True,inferSchema =True)
                    .withColumn("ingested_at", current_timestamp())
)

( bronze_ingestion.write.mode("overwrite")
         .format("delta")
         .saveAsTable("bronze_bookings")
)


## Task 6 - Silver: clean and conform
Build `silver_bookings` from bronze: keep only completed bookings and join the
hotel dimension to add `category`, `star_rating`, and the hotel name. Drop the
duplicate `city` from the hotel side so the join has a single `city`.

In [0]:
# TODO

from pyspark.sql.functions import col

silver_bookings_df = (spark.table("bronze_bookings").alias("b")
                                        .filter(col("status") =="completed")
                                        .join(hotels_df.alias("h") ,col("b.hotel_id") == col("h.hotel_id") ,"inner" )
                                        .drop(col("h.city"),col("h.hotel_id"))
                                        .select("booking_id","customer_id","hotel_id"
                                                ,"booking_date","city","nights","amount","status","h.hotel_name","h.category",
                                                "h.star_rating","ingested_at")
)   

In [0]:
(silver_bookings_df.write.mode("overwrite")
                        .format("delta")
                        .saveAsTable("silver_bookings")

)

## Task 7 - Gold: business-ready aggregate
From silver, build a `gold_city_revenue` Delta table: bookings and total revenue
per city, ordered by revenue.

In [0]:
# TODO
from pyspark.sql.functions import sum,count


gold_city_revenue_df = (spark.table("silver_bookings")
                              .groupBy("city")
                              .agg(
                                  count(col("booking_id")).alias("total_bookings"),
                                  sum(col("amount")).alias("total_revenue"))
                              .orderBy(col("total_revenue").desc())

)

(gold_city_revenue_df.write.mode("overwrite")
                        .format("delta")
                        .saveAsTable("gold_city_revenue")

)

## Task 8 - Incremental load with MERGE
You have today's batch in `updates_df` (150 changed bookings + 50 new ones).
`MERGE` it into your Delta table: update matched booking_ids, insert new ones, in
one command. Report the row count before and after (it should grow by the 50 new).

In [0]:
%sql
select count(*) as before_merge_count from bookings;


before_merge_count
12000


In [0]:
Base    = "/Volumes/workspace/default/staynest"

updates_df =(spark.read.csv(f"{Base}/bookings_updates.csv" ,header =True, inferSchema = True)
)

updates_df.createOrReplaceTempView("booking_updates")

In [0]:
spark.sql(f"""
    MERGE INTO bookings AS target
    USING booking_updates       AS source
        ON target.booking_id = source.booking_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select count(*) as after_merge_count from bookings;

after_merge_count
12050
